# Load public IBL trials

This notebook sketches the workflow for finding public International Brain Laboratory sessions with ONE, loading trial tables, and converting them into a psytrax data dictionary. The longer notebook lives at `examples/ibl_one_integration_walkthrough.ipynb`.

![IBL ONE to psytrax workflow](../_static/images/examples/ibl-integration-workflow.svg)

## Connect to public OpenAlyx

Install the IBL extra before running this locally: `pip install -e .[ibl]`.

In [ ]:
from one.api import ONE

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True,
)

In [ ]:
def search_subjects(query, limit=25):
    matches = one.alyx.rest(
        "subjects",
        "list",
        django=f"nickname__istartswith,{query}",
        limit=limit,
    )
    return [record["nickname"] for record in matches]


subject_matches = search_subjects("KS023")
subject_matches[:5]

## Convert trials into a psytrax dictionary

The important part is to make the response coding, signed contrast, reaction-time units, and session lengths explicit.

In [ ]:
import numpy as np


def ibl_trials_to_psytrax(trials, session_lengths):
    contrast_left = np.nan_to_num(trials["contrastLeft"], nan=0.0)
    contrast_right = np.nan_to_num(trials["contrastRight"], nan=0.0)
    signed_contrast = contrast_right - contrast_left

    choice_right = (np.asarray(trials["choice"]) == 1).astype(float)
    reaction_time = np.asarray(trials["response_times"]) - np.asarray(trials["stimOn_times"])

    valid = np.isfinite(signed_contrast) & np.isfinite(choice_right) & np.isfinite(reaction_time)
    return {
        "inputs": {"c": signed_contrast[valid]},
        "responses": choice_right[valid],
        "times": reaction_time[valid],
        "session_lengths": np.asarray(session_lengths),
    }